# 06 — Concentration Conditioning: Primary Test

First estimation of the core concentration–momentum relationship.

Primary: broad-market company-level Top-10 share at t -> value-weighted 12–2 momentum Q5-Q1 return realised in t+1. HAC6 is primary; HAC12 is robustness.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel, build_monthly_history_panel, build_market_state_panel
from src.concentration import calculate_monthly_concentration
from src.signals import attach_momentum_to_formation_universe, assign_signal_quintiles
from src.portfolio import monthly_rank_ic, quintile_returns, quintile_return_panel
from src.conditioning import merge_performance_and_concentration, primary_top10_spec, hhi_spec, expanding_concentration_regimes, regime_descriptives


## 1. Rebuild validated inputs


In [ ]:
RAW = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'
clean, _ = build_clean_monthly_panel(RAW, CleaningConfig())
history = build_monthly_history_panel(RAW)
market_state = build_market_state_panel(RAW)
conc = calculate_monthly_concentration(market_state)
mom = attach_momentum_to_formation_universe(clean, history)
mom = assign_signal_quintiles(mom)
rank_ic = monthly_rank_ic(mom)
qret = quintile_returns(mom)
vw = quintile_return_panel(qret, return_col='VW_Return')
ew = quintile_return_panel(qret, return_col='EW_Return')


## 2. Alignment check


In [ ]:
primary = merge_performance_and_concentration(vw, conc)
primary[['MthCalDt','Q5_minus_Q1','Top10Share','HHI']].tail(12)


## 3. PRIMARY — VW Q5−Q1 on Top-10 share, HAC6


In [ ]:
primary_hac6, _ = primary_top10_spec(primary, maxlags=6)
primary_hac6


## 4. HAC12 robustness


In [ ]:
primary_hac12, _ = primary_top10_spec(primary, maxlags=12)
primary_hac12


## 5. Time-trend robustness


In [ ]:
primary_trend_hac6, _ = primary_top10_spec(primary, maxlags=6, add_time_trend=True)
primary_trend_hac6


## 6. HHI robustness


In [ ]:
hhi_hac6, _ = hhi_spec(primary, maxlags=6)
hhi_hac6


## 7. Secondary — equal-weighted spread


In [ ]:
secondary_ew = merge_performance_and_concentration(ew, conc)
ew_hac6, _ = primary_top10_spec(secondary_ew, maxlags=6)
ew_hac6


## 8. Secondary — Rank IC


In [ ]:
secondary_ic = merge_performance_and_concentration(rank_ic[['MthCalDt','RankIC']], conc)
ic_hac6, _ = primary_top10_spec(secondary_ic, outcome_col='RankIC', maxlags=6)
ic_hac6


## 9. Descriptive point-in-time regimes


In [ ]:
regimes = expanding_concentration_regimes(conc, min_history=60)
regime_data = primary.merge(regimes[['MthCalDt','Prior_Q33','Prior_Q67','ConcentrationRegime']], on='MthCalDt', how='left', validate='one_to_one')
regime_data = regime_data.merge(rank_ic[['MthCalDt','RankIC']], on='MthCalDt', how='left', validate='one_to_one')
regime_descriptives(regime_data)


In [ ]:
regime_data['ConcentrationRegime'].value_counts(dropna=False)


## 10. Visual diagnostic only


In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(primary['Top10Share'], primary['Q5_minus_Q1'], alpha=0.45)
ax.set_title('VW momentum spread vs market concentration')
ax.set_xlabel('Top-10 company market-cap share')
ax.set_ylabel('Q5 − Q1 next-month return')
plt.show()
